***Movie recommendation system -- Data Cleaning***

In [32]:
#import section
import pandas as pd
from datetime import datetime, timezone

***Ratings of Movies data processing and cleaning***

In [33]:
df_ratings=pd.read_csv("H:\\CODINGS\\ML_projects\\movie_recom_system\\data\\raw\\mov_lens_small\\ratings.csv")
df_movies=pd.read_csv("H:\\CODINGS\\ML_projects\\movie_recom_system\\data\\raw\\mov_lens_small\\movies.csv")
df_links=pd.read_csv("H:\\CODINGS\\ML_projects\\movie_recom_system\\data\\raw\\mov_lens_small\\links.csv")
df_tags=pd.read_csv("H:\\CODINGS\\ML_projects\\movie_recom_system\\data\\raw\\mov_lens_small\\tags.csv")

# df_movies.head(5)

# df_links.head(5)
# df_movies.shape,df_tags.shape,df_links.shape
df_tags.drop(columns='userId',inplace=True)
df_tags.sort_values(by='movieId')

,movieId,tag,timestamp
2886,1,fun,1525286013
981,1,pixar,1137206825
629,1,pixar,1139045764
35,2,Robin Williams,1528843907
34,2,magic board game,1528843932
...,...,...,...
402,187595,star wars,1528934552
528,193565,comedy,1537098587
527,193565,anime,1537098582
530,193565,remaster,1537098592


In [34]:
#need to process the tags
# group by movie, then use join to combine tags into a single string while maintaining duplicates.
aggregated_tags = (df_tags.groupby(['movieId'])['tag'].apply(lambda x:', '.join(set(x))).reset_index())

#let's merge them all based on the movie id
merged_movie_links=(df_movies.merge(df_links, on=["movieId"], how='left').merge(aggregated_tags, on="movieId", how="left"))

#let's get the tags as well
merged_df=df_ratings.merge(merged_movie_links,on=["movieId"])

#as the timestap is not human readable , let's convert this into two different data(one will be utc and utc+6[for bd timezone])
utc_column=pd.to_datetime(df_ratings["timestamp"],utc=True,unit='s')
local_column=pd.to_datetime(df_ratings["timestamp"],utc=True,unit='s').dt.tz_convert('Asia/Dhaka')
merged_df.insert(loc=1, column="utc_column", value=utc_column)
merged_df.insert(loc=1, column="local_column", value=local_column)

# merged_df.to_csv("processed_merged_data.csv")
merged_df

,userId,local_column,utc_column,movieId,rating,timestamp,title,genres,imdbId,tmdbId,tag
0,1,2000-07-31 00:45:03+06:00,2000-07-30 18:45:03+00:00,1,4.0,964982703,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,114709,862.0,"fun, pixar"
1,5,2000-07-31 00:20:47+06:00,2000-07-30 18:20:47+00:00,1,4.0,847434962,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,114709,862.0,"fun, pixar"
2,7,2000-07-31 00:37:04+06:00,2000-07-30 18:37:04+00:00,1,4.5,1106635946,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,114709,862.0,"fun, pixar"
3,15,2000-07-31 01:03:35+06:00,2000-07-30 19:03:35+00:00,1,2.5,1510577970,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,114709,862.0,"fun, pixar"
4,17,2000-07-31 00:48:51+06:00,2000-07-30 18:48:51+00:00,1,4.5,1305696483,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,114709,862.0,"fun, pixar"
...,...,...,...,...,...,...,...,...,...,...,...
100831,610,2017-05-04 03:53:22+06:00,2017-05-03 21:53:22+00:00,160341,2.5,1479545749,Bloodmoon (1997),Action|Thriller,118745,30948.0,NaN
100832,610,2017-05-04 04:21:31+06:00,2017-05-03 22:21:31+00:00,160527,4.5,1479544998,Sympathy for the Underdog (1971),Action|Crime|Drama,66806,90351.0,NaN
100833,610,2017-05-09 01:50:47+06:00,2017-05-08 19:50:47+00:00,160836,3.0,1493844794,Hazard (2005),Action|Drama|Thriller,798722,70193.0,NaN
100834,610,2017-05-04 03:19:12+06:00,2017-05-03 21:19:12+00:00,163937,3.5,1493848789,Blair Witch (2016),Horror|Thriller,1540011,351211.0,NaN


In [36]:
#There are still some data left to preprocess like generes column data, we can make them a list.
#there are several methods to do this , i will be using the replace so that i'll not need a new column 
merged_df['genres']=merged_df['genres'].str.replace("|",",")
merged_df
#From the previous section we know that the merged dataset has some nan value at tag column. so let's dive on the cleaning


,userId,local_column,utc_column,movieId,rating,timestamp,title,genres,imdbId,tmdbId,tag
0,1,2000-07-31 00:45:03+06:00,2000-07-30 18:45:03+00:00,1,4.0,964982703,Toy Story (1995),"Adventure,Animation,Children,Comedy,Fantasy",114709,862.0,"fun, pixar"
1,5,2000-07-31 00:20:47+06:00,2000-07-30 18:20:47+00:00,1,4.0,847434962,Toy Story (1995),"Adventure,Animation,Children,Comedy,Fantasy",114709,862.0,"fun, pixar"
2,7,2000-07-31 00:37:04+06:00,2000-07-30 18:37:04+00:00,1,4.5,1106635946,Toy Story (1995),"Adventure,Animation,Children,Comedy,Fantasy",114709,862.0,"fun, pixar"
3,15,2000-07-31 01:03:35+06:00,2000-07-30 19:03:35+00:00,1,2.5,1510577970,Toy Story (1995),"Adventure,Animation,Children,Comedy,Fantasy",114709,862.0,"fun, pixar"
4,17,2000-07-31 00:48:51+06:00,2000-07-30 18:48:51+00:00,1,4.5,1305696483,Toy Story (1995),"Adventure,Animation,Children,Comedy,Fantasy",114709,862.0,"fun, pixar"
...,...,...,...,...,...,...,...,...,...,...,...
100831,610,2017-05-04 03:53:22+06:00,2017-05-03 21:53:22+00:00,160341,2.5,1479545749,Bloodmoon (1997),"Action,Thriller",118745,30948.0,NaN
100832,610,2017-05-04 04:21:31+06:00,2017-05-03 22:21:31+00:00,160527,4.5,1479544998,Sympathy for the Underdog (1971),"Action,Crime,Drama",66806,90351.0,NaN
100833,610,2017-05-09 01:50:47+06:00,2017-05-08 19:50:47+00:00,160836,3.0,1493844794,Hazard (2005),"Action,Drama,Thriller",798722,70193.0,NaN
100834,610,2017-05-04 03:19:12+06:00,2017-05-03 21:19:12+00:00,163937,3.5,1493848789,Blair Witch (2016),"Horror,Thriller",1540011,351211.0,NaN
